# Model Iteration with TPOT
TPOT seems very effective in finding a well-performing pipeline for stress/no-stress classification. In this script, we run TPOT for different datasets. We look at datasets created with FLIRT with the following window sizes and step sizes:

Window size: 10; step size: 1
Window size: 10; step size: 5
Window size: 10; step size: 10
Window size: 15; step size: 1
Window size: 15; step size: 5
Window size: 15; step size: 15
Window size: 30; step size: 1
Window size: 30; step size: 5
Window size: 30; step size: 10
The idea behind this is that it would be better to predict stress more quickly, i.e., based on fewer seconds of data - i.e., shorter window sizes. The step size indicated how many seconds we advance to calculate the next window of features. The idea in trying out different values here is to check if we might overfit when using very small step sizes.

The goal of this script is to get an idea about good pipeline and to get an idea of how long the window size and the step size should be.

# Imports


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from tpot import TPOTClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score, accuracy_score, ConfusionMatrixDisplay
from sklearn.metrics import PrecisionRecallDisplay
import matplotlib.pyplot as plt

import utils

# Functions

In [ ]:
def run_tpot(windowsize, stepsize, X_train, y_train, groups_train, X_test, y_test, groups_test, gens, pop_size, cvs):
    '''
    Takes the prepared data and group indication for training and testing a TPOT pipeline.
    Returns X_train, y_train, groups_train, X_test, y_test, groups_test.
    Prints out info during creation.
    '''

    tpot = TPOTClassifier(generations=gens,
                      population_size=pop_size,
                      scoring='f1',
                      cv=cvs,
                      n_jobs=-1,
                      verbosity=2,
                      random_state=0)

    tpot.fit(X_train, y_train.values.ravel(), groups=groups_train.values.ravel())

    print(f"TPOT score (F1) on test data: {tpot.score(X_test, y_test.values.ravel()):.2f}")

    y_test_predict = tpot.predict(X_test)
    y_test_predict_proba = tpot.predict_proba(X_test)[:,1]

    print('accuracy: '+str(accuracy_score(y_test, y_test_predict)))
    print('precision: '+str(precision_score(y_test, y_test_predict)))
    print('recall: '+str(recall_score(y_test, y_test_predict)))
    print('f1_score: '+str(f1_score(y_test, y_test_predict)))
    print('roc_auc: '+str(roc_auc_score(y_test, y_test_predict)))
    print('Class distribution in test set: \n', y_test['label'].value_counts(normalize=True), '\n')
    print('average_precision: '+str(average_precision_score(y_test, y_test_predict_proba)))

    confusion_matrix(y_test, y_test_predict)

    prd = PrecisionRecallDisplay.from_predictions(y_test, y_test_predict_proba, name='Model')
    _ = prd.ax_.set_title('2-class Precision-Recall curve')
    print('The baseline to beat is the percentage of positive cases:', y_test['label'].value_counts(normalize=True)[1])
    plt.show()

    # store results
    filestring = 'tpot-exportet-wesad-'+str(windowsize)+'-'+str(stepsize)+'-'+str(gens)+'-'+str(pop_size)+'-'+str(cvs)
    tpot.export(filestring+'.py')
    print('Resulting pipeline stored to file: ', filestring)

    print('===============================================\n\n')

In [ ]:
def full_run_tpot(windowsize, stepsize, gens, pop_size, cvs):
    '''
    Reads data and runs tpot.
    '''

    X_train, y_train, groups_train, X_test, y_test, groups_test = utils.read_data(windowsize, stepsize)
    run_tpot(windowsize, stepsize, X_train, y_train, groups_train, X_test, y_test, groups_test, gens, pop_size, cvs)

# Run TPOT pipelines

In [ ]:
%%time
full_run_tpot(10, 1, 5, 20, 5)
full_run_tpot(10, 5, 5, 20, 5)
full_run_tpot(10, 10, 5, 20, 5)

full_run_tpot(15, 1, 5, 20, 5)
full_run_tpot(15, 5, 5, 20, 5)
full_run_tpot(15, 15, 5, 20, 5)

full_run_tpot(30, 1, 5, 20, 5)
full_run_tpot(30, 5, 5, 20, 5)
full_run_tpot(30, 10, 5, 20, 5)

## Interpretation
The results are:

Window-size; step size; F1 score

10-1: 0.8
10-5: 0.58
10-10: 0.7
15-1: 0.73
15-5: 0.8
15-15: 0.66
30-1: 0.87
30-5: 0.84
30-10: 0.77
Overall, we notice that the F1 score generally improves when going from 10 to 15 to 30 seconds window size. We do not observe overfitting with step size 1 second, rather, we see the best performance when using step size 1. In our initial TPOT pipeline test, we saw an F1 score of 0.88 for window size 60 and step size 10. Now, we have an F1 score of 0.87 for window size 30. My conclusion from these results here are that we continue working with step size 1 and with window size 30, because at this window size, the trade-off between F1 score and amount of data in seconds we need to make a prediction seem best.

Dicision: Step size 1, window size 30.

In the next step, we will conduct a hyperparameter search when building a ExtraTreesClassifier pipeline.